# ⏰ Scheduled Agents with Routines (Preview) 🏦

Welcome to the **Routines** tutorial! Some agent work shouldn't wait for a human to click "run" — compliance reviews, end-of-day reconciliations, and overnight reports need to happen *on a schedule*. **Foundry Routines** let you trigger an agent automatically on a cron schedule, a timer, or external events.

In this notebook we'll build a **Daily Benefits Compliance** reviewer for a retail bank and schedule it to run every morning:

1. **Initialize** an `AIProjectClient` with the **preview** surface enabled.
2. **Create a prompt agent** that reviews employee-benefits policies for compliance gaps.
3. **Schedule it** with `beta.routines` using a daily cron trigger (09:00 UTC).
4. **List** routines and inspect run history.
5. **Clean up** (and learn why scheduled routines need a ConnectorGateway to fully delete).

### ⚠️ Important Disclaimer
> This notebook is for educational and demonstration purposes only and is not financial, legal, or compliance advice. Always involve qualified compliance officers for real banking workflows.

> 🧪 **Preview:** Routines use the `project.beta.*` surface and require `allow_preview=True`. APIs may change.


## 🔐 Authentication Setup

This notebook uses **Entra ID only** (no keys). Make sure you're signed in with the Azure CLI before running the cells below:

```bash
az login
```

`DefaultAzureCredential` will pick up your CLI session automatically. The first call may take a few seconds while credentials resolve.


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    ScheduleRoutineTrigger,
    InvokeAgentResponsesApiRoutineAction,
)

# Load environment variables from the repo-root .env
load_dotenv()

project_endpoint = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
print(f"🔗 Project endpoint: {project_endpoint}")
print(f"🤖 Model deployment: {model_deployment}")

# allow_preview=True unlocks the project.beta.* surface (routines live there)
project = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(exclude_managed_identity_credential=True),
    allow_preview=True,
)
print("✅ AIProjectClient ready (preview surface enabled)")

## 1. Create the agent the routine will drive 🤖

A routine *invokes an agent*, so we need one first. We'll create a **prompt agent** — the "Daily Benefits Compliance" reviewer — using `agents.create_version` with a `PromptAgentDefinition`. Its job: scan the bank's employee-benefits offerings and flag anything that needs a compliance officer's attention.


In [ ]:
AGENT_NAME = "daily-benefits-compliance"

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=(
            "You are the Daily Benefits Compliance reviewer for a retail bank. "
            "Each day, review the bank's employee-benefits policies (retirement matching, "
            "health plans, and savings programs) for regulatory and policy compliance gaps. "
            "Produce a concise bullet-point report: flag missing disclosures, expiring "
            "enrollment windows, and contribution limits that exceed regulation. "
            "Always recommend that a licensed compliance officer confirm any findings."
        ),
    ),
)
print(f"🎉 Agent ready: {agent.name} (version {agent.version})")


## 2. Schedule it with a Routine ⏰

A **routine** ties a **trigger** to an **action**. We'll use a `ScheduleRoutineTrigger` with a daily cron expression (`0 9 * * *` = 09:00 every day) and an `InvokeAgentResponsesApiRoutineAction` that calls our agent with a fixed prompt. A routine's **action is immutable after creation**, so we use a fresh, timestamped name on each run.

| Cron | Meaning |
|------|---------|
| `0 9 * * *` | Every day at 09:00 UTC |
| `0 */4 * * *` | Every 4 hours |
| `0 9 * * 1-5` | Weekdays at 09:00 |


In [ ]:
from datetime import datetime, timezone

# A routine's action is immutable after creation, so use a fresh name per run
ROUTINE_NAME = f"daily-benefits-compliance-{datetime.now(timezone.utc):%Y%m%d%H%M%S}"

routine = project.beta.routines.create_or_update(
    ROUTINE_NAME,
    description="Daily 09:00 UTC compliance review of employee-benefits policies.",
    enabled=True,
    triggers={
        "daily": ScheduleRoutineTrigger(cron_expression="0 9 * * *", time_zone="UTC"),
    },
    action=InvokeAgentResponsesApiRoutineAction(
        agent_name=AGENT_NAME,
        input="Run today's employee-benefits compliance review and summarize any gaps.",
    ),
)
print(f"⏰ Routine '{routine.name}' scheduled | enabled={routine.enabled}")
print(f"   Triggers: {list(routine.triggers)} | action -> {routine.action.agent_name}")


## 3. List routines & inspect run history 📋

Once registered, the routine runs on its own. `beta.routines.list()` shows everything scheduled in the project, and `get` returns details for one. Run history (`list_runs`) is empty until the first scheduled fire, so it may show no runs right after creation.


In [ ]:
print("📋 Routines in this project:")
for r in project.beta.routines.list():
    print(f"  • {r.name} | enabled={r.enabled} | {r.description}")

# Run history is empty until the schedule first fires
try:
    runs = list(project.beta.routines.list_runs(ROUTINE_NAME))
    print(f"\n🏃 {ROUTINE_NAME}: {len(runs)} run(s) so far")
    for run in runs[:3]:
        print(f"   - {run.id}: {run.status} ({run.trigger_type})")
except Exception as e:
    print(f"\nℹ️ No run history yet (routine fires on schedule): {type(e).__name__}")


## 4. Clean up 🧹

To avoid an agent firing every morning, we **disable** the routine. Note: scheduled routines can't be fully **deleted** without a configured **ConnectorGateway**, so `delete` may raise — we wrap it in `try/except`, disable as the fallback, and remove the agent. Cleanup never fails the notebook.


In [ ]:
# Disable first so it stops firing even if delete is unavailable
try:
    project.beta.routines.disable(ROUTINE_NAME)
    print(f"🛑 Disabled routine '{ROUTINE_NAME}'")
except Exception as e:
    print(f"ℹ️ Disable skipped: {type(e).__name__}")

# Scheduled routines need a ConnectorGateway to delete; tolerate failure
try:
    project.beta.routines.delete(ROUTINE_NAME)
    print(f"🗑️ Deleted routine '{ROUTINE_NAME}'")
except Exception as e:
    print(f"ℹ️ Routine left disabled (delete needs a ConnectorGateway): {type(e).__name__}")

try:
    # [kept-in-foundry] project.agents.delete_version(AGENT_NAME, agent.version)
    print(f"🗑️ Deleted agent '{AGENT_NAME}' v{agent.version}")
except Exception as e:
    print(f"ℹ️ Agent cleanup skipped: {type(e).__name__}")

project.close()
print("✅ Cleanup complete")

## 📚 References

- [Routines in Foundry Agent Service (preview)](https://learn.microsoft.com/azure/foundry/agents/concepts/routines) — concept: scheduled/triggered automation
- [Automate agents with routines (preview)](https://learn.microsoft.com/azure/foundry/agents/how-to/use-routines) — create and run a routine, step by step
- [azure-ai-projects (Python) reference](https://learn.microsoft.com/python/api/overview/azure/ai-projects-readme?view=azure-python) — `beta.routines` / `beta.schedules` APIs
- [Quickstart: Create a prompt agent](https://learn.microsoft.com/azure/foundry/agents/quickstarts/prompt-agent) — the agent a routine drives
> 💡 Tip: search the official docs live from your agent via the **Microsoft Learn MCP** at `https://learn.microsoft.com/api/mcp`.